# MusicGen Melody Conditioning

MusicGen-melody can generate music conditioned on both text AND a melody reference audio.
It extracts melodic contour from the reference and generates new audio that follows the
same melody with different instrumentation/style.

This demonstrates a form of **musical style transfer**: keep the melody, change everything else.

In [ ]:
!pip install transformers torch torchaudio scipy matplotlib IPython librosa numpy

In [ ]:
from transformers import AutoProcessor, MusicgenForConditionalGeneration
import torch
import torchaudio
import librosa
import numpy as np
import IPython.display as ipd
import matplotlib.pyplot as plt
import scipy.io.wavfile

processor = AutoProcessor.from_pretrained("facebook/musicgen-melody")
model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-melody")
model = model.to("cuda" if torch.cuda.is_available() else "cpu")
print(f"Model loaded on {next(model.parameters()).device}")

sampling_rate = model.config.audio_encoder.sampling_rate
print(f"Sampling rate: {sampling_rate} Hz")

## Load Reference Audio

We synthesize a simple melody as our reference. You can also load your own audio file.

In [ ]:
def synthesize_melody(notes, durations, sr=32000):
    """Synthesize a simple sine-wave melody from MIDI note numbers and durations."""
    audio = []
    for note, dur in zip(notes, durations):
        t = np.linspace(0, dur, int(sr * dur), endpoint=False)
        freq = 440.0 * 2 ** ((note - 69) / 12.0)
        # Sine with simple amplitude envelope
        envelope = np.minimum(t / 0.02, 1.0) * np.minimum((dur - t) / 0.05, 1.0)
        tone = 0.5 * np.sin(2 * np.pi * freq * t) * envelope
        audio.append(tone)
    return np.concatenate(audio).astype(np.float32)

# "Twinkle Twinkle Little Star" in C major
notes =     [60, 60, 67, 67, 69, 69, 67, 65, 65, 64, 64, 62, 62, 60]
durations = [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 1.0, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 1.0]

sr = 32000
melody_audio = synthesize_melody(notes, durations, sr=sr)

print(f"Melody duration: {len(melody_audio) / sr:.1f}s")
print("Reference melody:")
ipd.display(ipd.Audio(melody_audio, rate=sr))

# Visualize
plt.figure(figsize=(12, 3))
time = np.arange(len(melody_audio)) / sr
plt.plot(time, melody_audio, linewidth=0.5)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Reference Melody Waveform")
plt.tight_layout()
plt.show()

## Melody-Conditioned Generation

We pass both a text prompt and the reference melody audio. MusicGen-melody extracts
the melodic contour and generates new audio that follows it.

In [ ]:
text = "Orchestral arrangement with strings and brass"

inputs = processor(
    text=[text],
    audio=melody_audio,
    sampling_rate=sr,
    padding=True,
    return_tensors="pt"
)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    audio = model.generate(**inputs, max_new_tokens=512)

print(f"Text: {text}")
print(f"Generated duration: {audio.shape[-1] / sampling_rate:.1f}s")
ipd.display(ipd.Audio(audio[0].cpu().numpy(), rate=sampling_rate))

## Style Transfer

Same reference melody, different text prompts — this creates different "arrangements"
of the same melodic material.

In [ ]:
style_prompts = [
    "Acoustic guitar fingerpicking in a folk style",
    "Electronic synth with arpeggiated chords and reverb",
    "Jazz piano with a walking bass and light drums",
    "8-bit chiptune video game music",
]

generated_styles = []

for prompt in style_prompts:
    inputs = processor(
        text=[prompt],
        audio=melody_audio,
        sampling_rate=sr,
        padding=True,
        return_tensors="pt"
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512)

    generated_styles.append(out[0].cpu().numpy())
    print(f"\nStyle: {prompt}")
    ipd.display(ipd.Audio(out[0].cpu().numpy(), rate=sampling_rate))

## Comparison

Compare the original melody alongside all generated style variations.

In [ ]:
fig, axes = plt.subplots(len(style_prompts) + 1, 1, figsize=(14, 3 * (len(style_prompts) + 1)))

# Original melody spectrogram
axes[0].specgram(melody_audio, Fs=sr, NFFT=1024, noverlap=512, cmap="magma")
axes[0].set_title("Original Reference Melody")
axes[0].set_ylabel("Freq (Hz)")
axes[0].set_ylim(0, 4000)

# Generated style spectrograms
for i, (prompt, gen_audio) in enumerate(zip(style_prompts, generated_styles)):
    audio_np = gen_audio.squeeze()
    axes[i + 1].specgram(audio_np, Fs=sampling_rate, NFFT=2048, noverlap=1024, cmap="magma")
    axes[i + 1].set_title(f"Style: {prompt}")
    axes[i + 1].set_ylabel("Freq (Hz)")
    axes[i + 1].set_ylim(0, 8000)

axes[-1].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()

# Save all outputs
import os
os.makedirs("outputs", exist_ok=True)
for i, (prompt, gen_audio) in enumerate(zip(style_prompts, generated_styles)):
    audio_np = gen_audio.squeeze()
    audio_int16 = (audio_np * 32767).astype(np.int16)
    filename = f"outputs/melody_style_{i+1}.wav"
    scipy.io.wavfile.write(filename, sampling_rate, audio_int16)
    print(f"Saved {filename} — '{prompt}'")